# Module 1: What you are building, and how to see inside it

You run the prototype agent once and look inside it in two ways: the raw request and response of each model call, and the run's trace in MLflow. Every later module comes back to these two: the payload and the trace.

Some cells end with a question to think through before you go on. There's nothing to type or hand in, and the answer is in a collapsed block right under the question.

In [ ]:
# Load the lab's notebook helpers (the lab. calls in later cells) and move into agent/,
# so the commands below run there.
import labnb as lab
%cd ../agent

## Start from module 1

The cell below puts the agent back where module 1 starts, using [`reset.py`](../agent/reset.py), the
lab's reset script. On a fresh lab it changes nothing. It's there so that running this notebook again
always starts from the same place. Before it replaces anything you edited, it copies your edits to
`agent/.reset-backup/`, so nothing you wrote is lost. The file browser hides that folder because its
name starts with a dot. Once a reset has saved edits, a cell with `!ls .reset-backup` lists the backups.

In [ ]:
# Put agent/ back in module 1's starting state, copying any edits you made to .reset-backup/ first.
!python reset.py 1

## 1. Meet the agent

The agent answers field service questions for Kestrel Industrial, a fictional maker of industrial
cooling equipment. It lives in the lab folder in your workbench, `agent-antipatterns-lab`, which has
the two parts you'll use:

```text
agent-antipatterns-lab/
├── notebooks/         the modules, one notebook each
└── agent/             the agent you'll fix
    ├── agent.py       the loop
    ├── tools.py       the tools the model is offered
    ├── config.toml    which model endpoint it calls
    ├── runs/          the payloads each run saves (created by the first run)
    └── ...            other files you'll meet in later modules
```

- [`agent.py`](../agent/agent.py) is the agent itself: the loop that sends your question to the model
  and runs the tools the model asks for.
- [`tools.py`](../agent/tools.py) holds the agent's two tools, `find_part` and `check_stock`, and the
  description of each one that the model is given.
- [`config.toml`](../agent/config.toml) is the agent's settings file. It says which model endpoint the
  agent calls, and holds the system prompt that tells the model its job.
- `runs/` is where the agent saves each run's payloads.

You don't need to open these files now. The notebook shows you the parts that matter as you go. The
first code cell above moved into `agent/`, so commands such as `python agent.py` run there.

The agent sends your question to the model, along with a list of tools the model
may use. The model can't run a tool itself. If it needs one, it replies with a structured request in a
field called `tool_calls`, naming the tool and its arguments. `agent.py` runs that tool from
`tools.py` and calls the model again. That cycle is the loop. When the model
replies without asking for a tool, the agent takes the reply as the answer and prints it.

The model runs on the cluster, not in your workbench. It's Qwen3-8B, served by vLLM through
OpenShift AI's model serving. The lab has two endpoints serving the same
model, and `config.toml` points the agent at one of them.

Each run leaves two records of what the agent did: the raw payloads under `agent/runs/`, and a trace in MLflow.

```mermaid
flowchart LR
  Q["Your question"] --> A["agent.py: the loop"]
  A -- "question and tool list" --> M["Model: qwen3-8b-a on vLLM"]
  M -- "an answer, or a tool_calls request" --> A
  A -- "runs the requested tool" --> T["tools.py: find_part, check_stock"]
  T -- "result" --> A
  A --> ANS["The answer it prints"]
  A -. "payloads" .-> R["agent/runs/"]
  A -. "one trace per run" .-> F["MLflow"]
```

The cell below prints the endpoint the agent starts on. Remember its name, `qwen3-8b-a`. Module 2
compares it with the other endpoint.

In [ ]:
# Print the model endpoint config.toml points the agent at.
print(lab.endpoint())

## 2. Run it once

A field engineer is with a customer whose K-400 shows a coolant pressure fault, and needs the part
number for a replacement coolant pump. The engineer has reported that the agent doesn't give the
right answers to questions like this one, and you're going to help fix it. The lab keeps its test questions in
[`questions.py`](../agent/questions.py), a list of named questions, and calls this one `m1`. The
command below asks it with `--q m1`.

The question needs a part number, and only `find_part` has it, because Kestrel's parts aren't in any
model's training data.

The cell runs the agent, then shows a table of the runs you've made in this notebook. Near the end, the
agent's output says where it saved the payloads and suggests `python payloads.py`. If MLflow recorded
the run, its last line links to the trace. You don't need to act on either yet. Step 3 runs
`payloads.py` for you, and step 4 opens the trace.

In [ ]:
# Ask the agent the lab's question 'm1':
#   "A customer's K-400 is showing a coolant pressure fault. What's the replacement part number
#   for the coolant pump?"
!python agent.py --q m1
# Show a table of this notebook's runs, one row each:
#   endpoint: which model endpoint answered
#   finish_reason (call 1): how the first model call ended
#   tool_calls (call 1): the tools the first model call asked for
#   turns: how many times the agent called the model
#   answer: what the agent printed
lab.evidence(columns=("endpoint", "finish_reason (call 1)", "tool_calls (call 1)", "turns", "answer"))

**Your read:** look at the answer. Did the agent look anything up? Did anything report a problem? To
check, read the `tool_calls (call 1)` and `turns` columns, not the answer text.

<details><summary>Check your answer</summary>

No, it looked nothing up. `tool_calls (call 1)` is `null` or `[]`, which means the model asked for no
tool, and 1 turn means the agent called the model only once. The answer names `find_part`, but that's
text the model wrote, not a tool the agent ran. Nothing reported a problem either: there's no error,
and the run ended normally. Problems like this don't show up as errors, which is why the lab checks what the agent
actually did. Step 3 shows where the tool call went.
</details>

## 3. Read the payload

Each time the agent calls the model, it saves two files under `agent/runs/`: the request it sent and
the response that came back. Together they're the payload. The next cell runs
[`payloads.py`](../agent/payloads.py), the lab's payload viewer, which prints a short digest of each
model call. Here's what its lines mean:

- `request`: how many messages the agent sent, and which tools it declared. Here that's 2 messages:
  the system prompt from `config.toml`, which tells the model its job, and your question. Declared
  tools are the ones the model may ask the agent to run. The trace calls the same list
  `tools_offered`.
- `finish_reason`: why the model stopped writing. `stop` means it finished its answer. `tool_calls`
  means it stopped to ask the agent to run a tool.
- `tool_calls`: the tools the model asked for, as structured data the agent can act on, printed as
  JSON on the lines under it. `null` or `[]` means it asked for none.
- `content`: the text the model wrote, printed on the lines under it with its line breaks. `null`
  means it wrote none.

The table's *call 1*, the digest's *model call 01* and the trace's `model_call_1` are the same request.

The digest ends by saying how to read the full JSON files, and repeats the trace link if there is one.
You don't need the raw files for this module.

For comparison, here's a model call that does ask for a tool, from a run in module 2 (the id is
shortened). `arguments` arrives as a string of JSON inside the JSON, which is why its quote marks
have backslashes in front of them:

```text
response: finish_reason = tool_calls
          tool_calls    =
            [
              {
                "id": "chatcmpl-tool-…",
                "type": "function",
                "function": {
                  "name": "find_part",
                  "arguments": "{\"query\": \"coolant pump\", \"product\": \"K-400\"}"
                }
              }
            ]
          content       = null
```

In [ ]:
# Print a digest of each model call in the newest run: what the agent sent and what came back.
!python payloads.py

In [ ]:
# Show the content of your run's first model call on its own: the text the agent took as its answer.
lab.content()

Now compare your run with the example. Your request declared the tools. In the response,
`tool_calls` is empty and `finish_reason` is `stop`. The request to call `find_part` is there, but it
arrived in `content`, the text field, not in `tool_calls`.

`agent.py` only looks at `tool_calls` to decide whether to run a tool. It found none, so it took
`content` as the final answer and printed it. That's why the answer you saw is a tool call that never
ran. Module 2 works out why the call
ended up in `content`.

The next cell shows the loop, the `run_agent` function in `agent.py`. You don't need to follow every
line. The `with tracing.span(...)` line makes the whole run one trace. The `for turn` loop calls the model, then
either stops with the answer or runs each tool the model asked for and goes round again. It has two other ways out: the words "final
answer" in the model's text, and running out of turns. Module 3 comes back to both. The line
`if not message.tool_calls:` is where the agent decides the model has answered. It records how the run
ended as `plain_answer`, which you'll see again in the trace.

In [ ]:
# Show run_agent from agent.py: the loop that calls the model and runs the tools it asks for.
lab.show_code("agent.py", "run_agent")

## 4. Your first trace

MLflow is an open-source tool for tracking machine learning work, and it runs as part of OpenShift AI.
This lab uses it to record what the agent does. Every run of the agent sends MLflow a **trace**, a
record of each step that you can open in your browser. MLflow keeps each participant's traces separate, in an
MLflow *workspace* named after your project, and the links below already open yours. Inside it,
MLflow groups your traces in an *experiment*, its word for a set of related runs, called
`kestrel-agent`. If MLflow asks you to log in, use the username and password from your lab's
provisioning details. If the Notebook tab asked you to log in, they're the same ones.

A trace is a tree of **spans**, one per step:

- `agent_run` is the whole run. Its outputs say how the run ended and how many tools it called.
  `ended_by: plain_answer` means the model replied without asking for a tool, so the agent stopped.
  `tool_call_count: 0` means it ran no tools.
- `model_call_1` is the agent's first request to the model. Its **Inputs** hold the messages and the
  tools offered. Its **Outputs** hold `finish_reason`, `tool_calls` and `content`.
- A tool the agent actually runs gets its own span under `agent_run`. This run should have none.

The first link below opens this run's trace. Look for its spans shown as a tree, with `agent_run` at
the top and `model_call_1` under it. Click `model_call_1` and look for its outputs, which may sit
under a tab of their own. Compare the `content` there with the `content` printed in step 3.
MLflow may show the line breaks as `\n` and put a backslash before each quote mark, and that's still
the same text. What should match is the `<tool_call>` block that names `find_part` and its arguments. The second link opens the list of every trace in the
experiment, one row per run. That list holds every run you make in this lab, newest first, so your
latest run is at the top. To see one question's runs, filter by the `question` tag, for example `m1`.
If you open MLflow some other way, choose the workspace `user-<your id>-app` and the experiment
`kestrel-agent`.

**This is the module's success signal:** the text the model wrote, `content`, appears in two places:
the saved payload and the `model_call_1` outputs in the MLflow trace. It's the same in both.

In [ ]:
# Link to the newest run's trace in MLflow, then to the list of every trace in the experiment.
lab.trace_link()
lab.links(("Every trace in this experiment", lab.experiment_url()))

## 5. The rule for the rest of the lab

From here on, every module ends by checking what the agent actually did, not what its answer says:
in modules 1 and 2 that's mainly the payload, and from module 3 on it's mainly the trace, which you've
now seen. You'll learn to
read the number of turns, the order of the tool calls and, from module 4, the category each tool
result reports.

## If the result differs

If the answer holds a real part number, or `tool_calls (call 1)` names a tool, the agent isn't on the
endpoint module 1 starts on. Check that the endpoint cell printed `qwen3-8b-a`. If it didn't, run the
reset cell at the top of the notebook again, then the cells after it.

If the agent cell ends in an error instead of an answer, such as a timeout or a connection error, it
couldn't reach the model endpoint. That's a problem with the lab environment, not the failure this
module shows. Wait a minute and run the cell again.

If the agent printed "No trace recorded: MLflow couldn't be reached", your payloads are still saved,
so steps 2 and 3 still work, but the trace link says "not available yet". Run the agent cell again to
get a trace.

The table, the `content` cell and the trace link only show runs made since the notebook's kernel
started. If you restarted the kernel, or the workbench restarted, they say "No runs in this notebook
yet", "No runs yet" or "not available yet", even though your earlier run is saved. Run the first code
cell again, then the agent cell and the cells after it.

If a cell says something like `module 'labnb' has no attribute ...`, which can happen after the lab
files are updated, choose **Kernel > Restart Kernel** and run the notebook again from the first cell.